In [0]:
# Sample data with nulls and duplicates for Data Quality checks
data = [
    (1, "Alice", "IT", 29, 55000),
    (2, "Ben", "HR", 34, 48000),
    (3, "Cara", "IT", 41, 62000),
    (4, "Dev", "Finance", 26, 45000),
    (5, "Ella", "HR", 31, 51000),
    (3, "Cara", "IT", 41, 62000),      # duplicate ID (3)
    (6, "Frank", None, 38, 68000),      # null department
    (7, "Grace", "Finance", None, 47000),  # null age
    (8, "Henry", "HR", 44, None),      # null salary
    (9, None, "IT", 33, 59000),        # null name
    (3, "Cara", "IT", 41, 62000)       # another duplicate ID (3)
]
columns = ["id", "name", "department", "age", "salary"]

df = spark.createDataFrame(data, columns)
print(f"Dataset created with {df.count()} rows")
df.display()

In [0]:

# Aggregations the whole Dataframe without grouping
from pyspark.sql.functions import count, avg, min, max
df.select(count("*")).display()
df.select(avg("salary")).display()
df.select(min("salary")).display()
df.select(max("salary")).display()

In [0]:

# Aggregations with groupBy
df.groupBy("department").count().display()
df.groupBy("department").avg("salary").display()
df.groupBy("department").min("salary").display()
df.groupBy("department").max("salary").display()



In [0]:
# Multiple aggregations with agg() and alias()
# Note: alias() renames the output column — without it, PySpark generates a default name like avg(salary), which is technically correct but awkward to reference later. Get in the habit of aliasing every aggregation from today onward.
from pyspark.sql.functions import count, avg, min, max, col
df.groupBy("department").agg(count("*").alias("employee_count"),
                             avg("salary").alias("average_salary"),
                             min("salary").alias("min_salary"),
                             max("salary").alias("max_salary")).display()


In [0]:
# Aggregations for Data Quality Checks (why this day matters most)
from pyspark.sql.functions import count, col, countDistinct, isnan, when

# 1. How many total rows?
print("=== 1. Total Rows ===")
df.select(count("*").alias("total_rows")).display()

# 2. How many nulls in a column?
print("\n=== 2. Null Count per Column ===")
df.select(
    count(when(col("name").isNull(), 1)).alias("nulls_in_name"),
    count(when(col("department").isNull(), 1)).alias("nulls_in_department"),
    count(when(col("age").isNull(), 1)).alias("nulls_in_age"),
    count(when(col("salary").isNull(), 1)).alias("nulls_in_salary")
).display()

# 3. What percentage of rows are null?
print("\n=== 3. Null Percentage ===")
total_count = df.count()
df.select(
    (count(when(col("name").isNull(), 1)) / total_count * 100).alias("name_null_pct"),
    (count(when(col("department").isNull(), 1)) / total_count * 100).alias("dept_null_pct"),
    (count(when(col("age").isNull(), 1)) / total_count * 100).alias("age_null_pct"),
    (count(when(col("salary").isNull(), 1)) / total_count * 100).alias("salary_null_pct")
).display()

# 4. How many duplicate IDs?
print("\n=== 4. Duplicate ID Check ===")
df.groupBy("id").count().filter(col("count") > 1).display()

# 5. What are all unique values in a column?
print("\n=== 5. Unique Values in Department Column ===")
df.select("department").distinct().display()

print("\n=== 5b. Count of Unique Values ===")
df.select(countDistinct("department").alias("unique_departments")).display()

